# Model Training & Evaluation

## Supply Chain Late Delivery Prediction

---

### Overview

Train and evaluate classification models to predict late deliveries.

### Modeling Strategy

| Phase | Models | Purpose |
|-------|--------|----------|
| 1. Baseline | Logistic Regression | Establish minimum performance |
| 2. Tree Models | Decision Tree, Random Forest | Capture non-linear patterns |
| 3. Boosting | XGBoost, LightGBM, CatBoost | State-of-the-art performance |
| 4. Best Model | Selected champion | Final evaluation & SHAP |

### Evaluation Metrics

| Metric | Why It Matters |
|--------|----------------|
| **F1 Score** | Balances precision and recall for imbalanced classes |
| **ROC-AUC** | Overall discriminative ability |
| **Precision** | Minimize false positives (avoid unnecessary interventions) |
| **Recall** | Minimize false negatives (catch actual late deliveries) |

---

In [1]:
# ============================================================
# SETUP
# ============================================================
import sys
import warnings
import time
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import plotly.io as pio
pio.templates.default = "plotly_white"

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load modern boosting libraries
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("XGBoost available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
    print("LightGBM available")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available")

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
    print("CatBoost available")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available")

XGBoost available
LightGBM available
CatBoost available


---

## 1. Load Features & Split Data

**Talking Point**: "We use the engineered features from the previous notebook with stratified train-test split."

In [3]:
# ============================================================
# LOAD DATA
# ============================================================
from src.data.preprocess import load_or_preprocess
from src.features.build_features import build_features_pipeline

# Load and build features
df = load_or_preprocess()
X, y = build_features_pipeline(df)

print(f"\nFeature matrix: {X.shape[0]:,} samples x {X.shape[1]} features")
print(f"Target: {y.sum():,} late ({y.mean()*100:.1f}%) / {(y==0).sum():,} on-time ({(1-y.mean())*100:.1f}%)")

📂 Loading latest file: /Users/unclesam/Projects/supply-chain-ml-project/data/interim/cleaned_data_20251205_0057.parquet
✅ Loaded cached preprocessed data from data/interim
FEATURE ENGINEERING PIPELINE
⚠️  LEAKAGE PREVENTION ACTIVE
    Excluded columns: shipping_date_(dateorders), delivery_status, delivery_days, late_delivery_risk, days_for_shipping_(real), delivery_status_encoded
✅ Temporal features created: day_of_week, month, quarter, is_weekend, days_since_start
✅ Customer features created: order_count, lifetime_value
✅ Product features created: popularity, category_popularity, order_value, discount_rate
✅ Shipping features created: shipping_urgency, scheduled_shipping_days, region_country
✅ Financial features created: profit_margin_pct, sales_per_item, is_high_value
✅ Encoded 10 categorical features
   (Excluded 'delivery_status' to prevent leakage)

✅ Selected 26 features for classification
   (Verified: No leaky features included)

Feature matrix shape: (180519, 26)
Target distri

In [ ]:
# ============================================================
# TRAIN-TEST SPLIT
# ============================================================
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")
print(f"\nClass distribution preserved:")
print(f"   Train late rate: {y_train.mean()*100:.1f}%")
print(f"   Test late rate: {y_test.mean()*100:.1f}%")

Train set: 144,415 samples
Test set: 36,104 samples

Class distribution preserved:
   Train late rate: 54.8%
   Test late rate: 54.8%


---

## 2. Phase 1: Baseline Model

**Talking Point**: "We start with Logistic Regression as our baseline. This gives us a reference point for more complex models."

In [ ]:
# ============================================================
# PHASE 1: BASELINE (Logistic Regression)
# ============================================================

# Store results for comparison
results = {}

print("PHASE 1: BASELINE MODEL")
print("=" * 60)

# Train Logistic Regression
start_time = time.time()
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)
lr_model.fit(X_train, y_train)
lr_time = time.time() - start_time

# Predictions
y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Metrics
results['Logistic Regression'] = {
    'model': lr_model,
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr, average='weighted'),
    'precision': precision_score(y_test, y_pred_lr, average='weighted'),
    'recall': recall_score(y_test, y_pred_lr, average='weighted'),
    'roc_auc': roc_auc_score(y_test, y_proba_lr),
    'train_time': lr_time,
    'y_proba': y_proba_lr
}

print(f"\nLogistic Regression (trained in {lr_time:.1f}s):")
print(f"   Accuracy: {results['Logistic Regression']['accuracy']:.4f}")
print(f"   F1 Score: {results['Logistic Regression']['f1']:.4f}")
print(f"   ROC-AUC: {results['Logistic Regression']['roc_auc']:.4f}")

PHASE 1: BASELINE MODEL

Logistic Regression (trained in 3.7s):
   Accuracy: 0.6894
   F1 Score: 0.6871
   ROC-AUC: 0.7070


---

## 3. Phase 2: Tree-Based Models

**Talking Point**: "Tree-based models can capture non-linear relationships. Random Forest is particularly robust."

In [6]:
# ============================================================
# PHASE 2: TREE-BASED MODELS
# ============================================================

print("\nPHASE 2: TREE-BASED MODELS")
print("=" * 60)

# Decision Tree
print("\nTraining Decision Tree...")
start_time = time.time()
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
    class_weight='balanced'
)
dt_model.fit(X_train, y_train)
dt_time = time.time() - start_time

y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

results['Decision Tree'] = {
    'model': dt_model,
    'accuracy': accuracy_score(y_test, y_pred_dt),
    'f1': f1_score(y_test, y_pred_dt, average='weighted'),
    'precision': precision_score(y_test, y_pred_dt, average='weighted'),
    'recall': recall_score(y_test, y_pred_dt, average='weighted'),
    'roc_auc': roc_auc_score(y_test, y_proba_dt),
    'train_time': dt_time,
    'y_proba': y_proba_dt
}
print(f"   Decision Tree: F1={results['Decision Tree']['f1']:.4f}, ROC-AUC={results['Decision Tree']['roc_auc']:.4f} ({dt_time:.1f}s)")

# Random Forest
print("\nTraining Random Forest...")
start_time = time.time()
rf_model = RandomForestClassifier(
    n_estimators=100,  # Reduced for speed
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_time = time.time() - start_time

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

results['Random Forest'] = {
    'model': rf_model,
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf, average='weighted'),
    'precision': precision_score(y_test, y_pred_rf, average='weighted'),
    'recall': recall_score(y_test, y_pred_rf, average='weighted'),
    'roc_auc': roc_auc_score(y_test, y_proba_rf),
    'train_time': rf_time,
    'y_proba': y_proba_rf
}
print(f"   Random Forest: F1={results['Random Forest']['f1']:.4f}, ROC-AUC={results['Random Forest']['roc_auc']:.4f} ({rf_time:.1f}s)")


PHASE 2: TREE-BASED MODELS

Training Decision Tree...
   Decision Tree: F1=0.7026, ROC-AUC=0.7625 (0.7s)

Training Random Forest...
   Random Forest: F1=0.6974, ROC-AUC=0.7899 (1.6s)


---

## 4. Phase 3: Gradient Boosting Models

**Talking Point**: "Modern gradient boosting algorithms (XGBoost, LightGBM, CatBoost) typically achieve state-of-the-art performance on tabular data."

In [ ]:
# ============================================================
# PHASE 3: GRADIENT BOOSTING MODELS
# ============================================================

print("\nPHASE 3: GRADIENT BOOSTING MODELS")
print("=" * 60)

# XGBoost
if XGBOOST_AVAILABLE:
    print("\nTraining XGBoost...")
    start_time = time.time()
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    xgb_time = time.time() - start_time

    y_pred_xgb = xgb_model.predict(X_test)
    y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

    results['XGBoost'] = {
        'model': xgb_model,
        'accuracy': accuracy_score(y_test, y_pred_xgb),
        'f1': f1_score(y_test, y_pred_xgb, average='weighted'),
        'precision': precision_score(y_test, y_pred_xgb, average='weighted'),
        'recall': recall_score(y_test, y_pred_xgb, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_xgb),
        'train_time': xgb_time,
        'y_proba': y_proba_xgb
    }
    print(f"   XGBoost: F1={results['XGBoost']['f1']:.4f}, ROC-AUC={results['XGBoost']['roc_auc']:.4f} ({xgb_time:.1f}s)")

# LightGBM
if LIGHTGBM_AVAILABLE:
    print("\nTraining LightGBM...")
    start_time = time.time()
    lgbm_model = LGBMClassifier(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        n_jobs=-1,
        verbose=-1
    )
    lgbm_model.fit(X_train, y_train)
    lgbm_time = time.time() - start_time

    y_pred_lgbm = lgbm_model.predict(X_test)
    y_proba_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

    results['LightGBM'] = {
        'model': lgbm_model,
        'accuracy': accuracy_score(y_test, y_pred_lgbm),
        'f1': f1_score(y_test, y_pred_lgbm, average='weighted'),
        'precision': precision_score(y_test, y_pred_lgbm, average='weighted'),
        'recall': recall_score(y_test, y_pred_lgbm, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_lgbm),
        'train_time': lgbm_time,
        'y_proba': y_proba_lgbm
    }
    print(f"   LightGBM: F1={results['LightGBM']['f1']:.4f}, ROC-AUC={results['LightGBM']['roc_auc']:.4f} ({lgbm_time:.1f}s)")

# CatBoost
if CATBOOST_AVAILABLE:
    print("\nTraining CatBoost...")
    start_time = time.time()
    catboost_model = CatBoostClassifier(
        iterations=100,
        depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        auto_class_weights='Balanced',
        verbose=False
    )
    catboost_model.fit(X_train, y_train)
    catboost_time = time.time() - start_time

    y_pred_catboost = catboost_model.predict(X_test)
    y_proba_catboost = catboost_model.predict_proba(X_test)[:, 1]

    results['CatBoost'] = {
        'model': catboost_model,
        'accuracy': accuracy_score(y_test, y_pred_catboost),
        'f1': f1_score(y_test, y_pred_catboost, average='weighted'),
        'precision': precision_score(y_test, y_pred_catboost, average='weighted'),
        'recall': recall_score(y_test, y_pred_catboost, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_catboost),
        'train_time': catboost_time,
        'y_proba': y_proba_catboost
    }
    print(f"   CatBoost: F1={results['CatBoost']['f1']:.4f}, ROC-AUC={results['CatBoost']['roc_auc']:.4f} ({catboost_time:.1f}s)")


PHASE 3: GRADIENT BOOSTING MODELS

Training XGBoost...
   XGBoost: F1=0.7036, ROC-AUC=0.7823 (0.3s)

Training LightGBM...
   LightGBM: F1=0.6974, ROC-AUC=0.7801 (0.8s)

Training CatBoost...
   CatBoost: F1=0.6931, ROC-AUC=0.7625 (0.8s)


---

## 4. Phase 4: Hyperparameter Tuning

**Talking Point**: "We optimize the best base models using hyperparameter tuning to improve performance further."


In [ ]:
# ============================================================
# PHASE 4: HYPERPARAMETER TUNING
# ============================================================
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

print("\nPHASE 4: HYPERPARAMETER TUNING")
print("=" * 60)

# Use best base model for tuning (XGBoost based on previous results)
if XGBOOST_AVAILABLE and 'XGBoost' in results:
    print("\nTuning XGBoost hyperparameters...")

    # Define parameter grid (reduced for speed - can expand for production)
    xgb_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.15],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5]
    }

    # Use RandomizedSearchCV for faster tuning
    print("   Running RandomizedSearchCV (this may take a few minutes)...")
    start_time = time.time()

    xgb_base = XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric='logloss'
    )

    xgb_tuned = RandomizedSearchCV(
        estimator=xgb_base,
        param_distributions=xgb_param_grid,
        n_iter=20,  # Number of parameter settings sampled
        scoring='f1_weighted',
        cv=3,  # 3-fold CV for speed
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1
    )

    xgb_tuned.fit(X_train, y_train)
    tuning_time = time.time() - start_time

    print(f"\n   Best parameters: {xgb_tuned.best_params_}")
    print(f"   Best CV score: {xgb_tuned.best_score_:.4f}")
    print(f"   Tuning time: {tuning_time:.1f}s")

    # Evaluate tuned model
    y_pred_tuned = xgb_tuned.predict(X_test)
    y_proba_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

    results['XGBoost (Tuned)'] = {
        'model': xgb_tuned.best_estimator_,
        'accuracy': accuracy_score(y_test, y_pred_tuned),
        'f1': f1_score(y_test, y_pred_tuned, average='weighted'),
        'precision': precision_score(y_test, y_pred_tuned, average='weighted'),
        'recall': recall_score(y_test, y_pred_tuned, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_tuned),
        'train_time': tuning_time,
        'y_proba': y_proba_tuned,
        'best_params': xgb_tuned.best_params_
    }

    print(f"\n   Tuned XGBoost Performance:")
    print(f"      F1 Score: {results['XGBoost (Tuned)']['f1']:.4f}")
    print(f"      ROC-AUC: {results['XGBoost (Tuned)']['roc_auc']:.4f}")

    # Compare with base XGBoost
    improvement = results['XGBoost (Tuned)']['f1'] - results['XGBoost']['f1']
    if improvement > 0:
        print(f"      Improvement: +{improvement:.4f} F1 score")
    else:
        print(f"      Change: {improvement:.4f} F1 score")
else:
    print("XGBoost not available or not in results. Skipping tuning.")

# Tune LightGBM as well
if LIGHTGBM_AVAILABLE and 'LightGBM' in results:
    print("\nTuning LightGBM hyperparameters...")

    # Define parameter grid for LightGBM
    lgbm_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8, 10],
        'learning_rate': [0.05, 0.1, 0.15],
        'num_leaves': [15, 31, 50, 70],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0],
        'min_child_samples': [10, 20, 30]
    }

    # Use RandomizedSearchCV for faster tuning
    print("   Running RandomizedSearchCV (this may take a few minutes)...")
    start_time = time.time()

    lgbm_base = LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
        class_weight='balanced'
    )

    lgbm_tuned = RandomizedSearchCV(
        estimator=lgbm_base,
        param_distributions=lgbm_param_grid,
        n_iter=20,  # Number of parameter settings sampled
        scoring='f1_weighted',
        cv=3,  # 3-fold CV for speed
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1
    )

    lgbm_tuned.fit(X_train, y_train)
    tuning_time = time.time() - start_time

    print(f"\n   Best parameters: {lgbm_tuned.best_params_}")
    print(f"   Best CV score: {lgbm_tuned.best_score_:.4f}")
    print(f"   Tuning time: {tuning_time:.1f}s")

    # Evaluate tuned model
    y_pred_lgbm_tuned = lgbm_tuned.predict(X_test)
    y_proba_lgbm_tuned = lgbm_tuned.predict_proba(X_test)[:, 1]

    results['LightGBM (Tuned)'] = {
        'model': lgbm_tuned.best_estimator_,
        'accuracy': accuracy_score(y_test, y_pred_lgbm_tuned),
        'f1': f1_score(y_test, y_pred_lgbm_tuned, average='weighted'),
        'precision': precision_score(y_test, y_pred_lgbm_tuned, average='weighted'),
        'recall': recall_score(y_test, y_pred_lgbm_tuned, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_lgbm_tuned),
        'train_time': tuning_time,
        'y_proba': y_proba_lgbm_tuned,
        'best_params': lgbm_tuned.best_params_
    }

    print(f"\n   Tuned LightGBM Performance:")
    print(f"      F1 Score: {results['LightGBM (Tuned)']['f1']:.4f}")
    print(f"      ROC-AUC: {results['LightGBM (Tuned)']['roc_auc']:.4f}")

    # Compare with base LightGBM
    improvement = results['LightGBM (Tuned)']['f1'] - results['LightGBM']['f1']
    if improvement > 0:
        print(f"      Improvement: +{improvement:.4f} F1 score")
    else:
        print(f"      Change: {improvement:.4f} F1 score")
else:
    print("LightGBM not available or not in results. Skipping tuning.")


---

## 5. Phase 5: Ensemble Modeling

**Talking Point**: "We combine multiple models using ensemble techniques to leverage the strengths of different algorithms and improve robustness."


In [ ]:
# ============================================================
# PHASE 5: ENSEMBLE MODELING
# ============================================================
from sklearn.ensemble import VotingClassifier, StackingClassifier

print("\nPHASE 5: ENSEMBLE MODELING")
print("=" * 60)

# Select top performing models for ensemble (prefer tuned versions if available)
top_models = {}
# Prefer tuned models over base models
model_preferences = {
    'XGBoost': ['XGBoost (Tuned)', 'XGBoost'],
    'LightGBM': ['LightGBM (Tuned)', 'LightGBM'],
    'Random Forest': ['Random Forest']  # No tuned version yet
}

for base_name, pref_list in model_preferences.items():
    for pref_name in pref_list:
        if pref_name in results:
            # Use a descriptive name for the ensemble
            display_name = pref_name if pref_name != base_name else base_name
            top_models[display_name] = results[pref_name]['model']
            break

if len(top_models) >= 2:
    print(f"\nCreating ensemble from {len(top_models)} models: {list(top_models.keys())}")

    # 1. Voting Classifier (Soft Voting)
    print("\n1. Training Voting Classifier (Soft Voting)...")
    start_time = time.time()

    voting_clf = VotingClassifier(
        estimators=list(top_models.items()),
        voting='soft',  # Use probability predictions
        n_jobs=-1
    )
    voting_clf.fit(X_train, y_train)
    voting_time = time.time() - start_time

    y_pred_voting = voting_clf.predict(X_test)
    y_proba_voting = voting_clf.predict_proba(X_test)[:, 1]

    results['Voting Ensemble'] = {
        'model': voting_clf,
        'accuracy': accuracy_score(y_test, y_pred_voting),
        'f1': f1_score(y_test, y_pred_voting, average='weighted'),
        'precision': precision_score(y_test, y_pred_voting, average='weighted'),
        'recall': recall_score(y_test, y_pred_voting, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_voting),
        'train_time': voting_time,
        'y_proba': y_proba_voting
    }

    print(f"   Voting Ensemble: F1={results['Voting Ensemble']['f1']:.4f}, ROC-AUC={results['Voting Ensemble']['roc_auc']:.4f} ({voting_time:.1f}s)")

    # 2. Stacking Classifier
    print("\n2. Training Stacking Classifier...")
    start_time = time.time()

    # Use best tuned model or best base model as meta-learner
    # Prefer tuned models, then base models
    if 'XGBoost (Tuned)' in results:
        meta_learner = results['XGBoost (Tuned)']['model']
        meta_name = 'XGBoost (Tuned)'
    elif 'LightGBM (Tuned)' in results:
        meta_learner = results['LightGBM (Tuned)']['model']
        meta_name = 'LightGBM (Tuned)'
    elif 'XGBoost' in results:
        meta_learner = XGBClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
        meta_name = 'XGBoost'
    elif 'LightGBM' in results:
        meta_learner = LGBMClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
        meta_name = 'LightGBM'
    else:
        meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
        meta_name = 'LogisticRegression'

    stacking_clf = StackingClassifier(
        estimators=list(top_models.items()),
        final_estimator=meta_learner,
        cv=3,  # 3-fold CV for meta-learner training
        n_jobs=-1
    )
    stacking_clf.fit(X_train, y_train)
    stacking_time = time.time() - start_time

    y_pred_stacking = stacking_clf.predict(X_test)
    y_proba_stacking = stacking_clf.predict_proba(X_test)[:, 1]

    results['Stacking Ensemble'] = {
        'model': stacking_clf,
        'accuracy': accuracy_score(y_test, y_pred_stacking),
        'f1': f1_score(y_test, y_pred_stacking, average='weighted'),
        'precision': precision_score(y_test, y_pred_stacking, average='weighted'),
        'recall': recall_score(y_test, y_pred_stacking, average='weighted'),
        'roc_auc': roc_auc_score(y_test, y_proba_stacking),
        'train_time': stacking_time,
        'y_proba': y_proba_stacking,
        'meta_learner': meta_name
    }

    print(f"   Stacking Ensemble (meta: {meta_name}): F1={results['Stacking Ensemble']['f1']:.4f}, ROC-AUC={results['Stacking Ensemble']['roc_auc']:.4f} ({stacking_time:.1f}s)")

    print("\n✅ Ensemble models trained successfully!")
else:
    print("⚠️ Need at least 2 models for ensemble. Skipping ensemble modeling.")


---

## 6. Model Comparison

**Talking Point**: "Let's compare all models and identify the best performer."

In [8]:
# ============================================================
# MODEL COMPARISON
# ============================================================

# Create comparison dataframe
comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': res['accuracy'],
        'F1 Score': res['f1'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'ROC-AUC': res['roc_auc'],
        'Train Time (s)': res['train_time']
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('F1 Score', ascending=False)

print("\nMODEL COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))


MODEL COMPARISON
              Model  Accuracy  F1 Score  Precision   Recall  ROC-AUC  Train Time (s)
            XGBoost  0.707401  0.703556   0.744060 0.707401 0.782276        0.319086
      Decision Tree  0.706404  0.702615   0.742594 0.706404 0.762504        0.675471
      Random Forest  0.702858  0.697434   0.747126 0.702858 0.789935        1.629379
           LightGBM  0.702748  0.697378   0.746700 0.702748 0.780108        0.750078
           CatBoost  0.698953  0.693139   0.744382 0.698953 0.762532        0.767226
Logistic Regression  0.689370  0.687108   0.715353 0.689370 0.706983        3.734009


In [9]:
# Visualize model comparison
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>F1 Score by Model</b>', '<b>ROC-AUC by Model</b>')
)

# Sort by F1 for visualization
sorted_df = comparison_df.sort_values('F1 Score', ascending=True)

# F1 Score
fig.add_trace(
    go.Bar(
        y=sorted_df['Model'],
        x=sorted_df['F1 Score'],
        orientation='h',
        marker_color='#3498db',
        text=[f"{v:.3f}" for v in sorted_df['F1 Score']],
        textposition='outside',
        name='F1'
    ),
    row=1, col=1
)

# ROC-AUC
fig.add_trace(
    go.Bar(
        y=sorted_df['Model'],
        x=sorted_df['ROC-AUC'],
        orientation='h',
        marker_color='#2ecc71',
        text=[f"{v:.3f}" for v in sorted_df['ROC-AUC']],
        textposition='outside',
        name='ROC-AUC'
    ),
    row=1, col=2
)

fig.update_layout(
    height=400,
    title='<b>Model Performance Comparison</b>',
    showlegend=False
)
fig.update_xaxes(range=[0.5, 1.0], row=1, col=1)
fig.update_xaxes(range=[0.5, 1.0], row=1, col=2)
fig.show()

# Identify best model
best_model_name = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1 Score']
best_roc = comparison_df.iloc[0]['ROC-AUC']

print(f"\nBEST MODEL: {best_model_name}")
print(f"   F1 Score: {best_f1:.4f}")
print(f"   ROC-AUC: {best_roc:.4f}")


BEST MODEL: XGBoost
   F1 Score: 0.7036
   ROC-AUC: 0.7823


---

## 6. Best Model Detailed Evaluation

**Talking Point**: "Let's do a deep dive into our best model's performance."

In [10]:
# ============================================================
# BEST MODEL EVALUATION
# ============================================================

best_model = results[best_model_name]['model']
best_proba = results[best_model_name]['y_proba']
best_pred = (best_proba >= 0.5).astype(int)

# Classification Report
print(f"\nCLASSIFICATION REPORT: {best_model_name}")
print("=" * 60)
print(classification_report(y_test, best_pred, target_names=['On-Time', 'Late']))


CLASSIFICATION REPORT: XGBoost
              precision    recall  f1-score   support

     On-Time       0.63      0.87      0.73     16308
        Late       0.84      0.58      0.68     19796

    accuracy                           0.71     36104
   macro avg       0.73      0.72      0.71     36104
weighted avg       0.74      0.71      0.70     36104



In [11]:
# Confusion Matrix & ROC Curve
cm = confusion_matrix(y_test, best_pred)
fpr, tpr, thresholds = roc_curve(y_test, best_proba)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, best_proba)

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "heatmap"}, {"type": "scatter"}, {"type": "scatter"}]],
    subplot_titles=('<b>Confusion Matrix</b>', '<b>ROC Curve</b>', '<b>Precision-Recall Curve</b>')
)

# Confusion Matrix
fig.add_trace(
    go.Heatmap(
        z=cm,
        x=['Pred: On-Time', 'Pred: Late'],
        y=['Actual: On-Time', 'Actual: Late'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text:,}',
        textfont={'size': 14},
        showscale=False
    ),
    row=1, col=1
)

# ROC Curve
roc_auc = roc_auc_score(y_test, best_proba)
fig.add_trace(
    go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'ROC (AUC={roc_auc:.3f})',
        line=dict(color='#e74c3c', width=2)
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1],
        mode='lines',
        name='Random',
        line=dict(color='gray', dash='dash')
    ),
    row=1, col=2
)

# PR Curve
fig.add_trace(
    go.Scatter(
        x=recall_curve, y=precision_curve,
        mode='lines',
        name='PR Curve',
        line=dict(color='#2ecc71', width=2)
    ),
    row=1, col=3
)

fig.update_layout(
    height=350,
    title=f'<b>{best_model_name} - Detailed Performance</b>',
    showlegend=True
)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=2)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=2)
fig.update_xaxes(title_text='Recall', row=1, col=3)
fig.update_yaxes(title_text='Precision', row=1, col=3)
fig.show()

# Business interpretation
tn, fp, fn, tp = cm.ravel()
print(f"\nBUSINESS INTERPRETATION:")
print(f"   True Positives (correctly caught late): {tp:,}")
print(f"   False Negatives (missed late deliveries): {fn:,}")
print(f"   False Positives (unnecessary interventions): {fp:,}")
print(f"   True Negatives (correctly on-time): {tn:,}")


BUSINESS INTERPRETATION:
   True Positives (correctly caught late): 11,402
   False Negatives (missed late deliveries): 8,394
   False Positives (unnecessary interventions): 2,170
   True Negatives (correctly on-time): 14,138


---

## 7. Feature Importance

**Talking Point**: "Understanding which features drive predictions helps explain the model to stakeholders."

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

# Get feature importance from best model
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importance = np.abs(best_model.coef_[0])
else:
    importance = None

if importance is not None:
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importance
    }).sort_values('Importance', ascending=True)

    # Plot top 15 features
    top_features = importance_df.tail(15)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=top_features['Feature'],
        x=top_features['Importance'],
        orientation='h',
        marker_color='#3498db',
        text=[f"{v:.3f}" for v in top_features['Importance']],
        textposition='outside'
    ))

    fig.update_layout(
        title=f'<b>Top 15 Feature Importances - {best_model_name}</b>',
        xaxis_title='Importance',
        height=500,
        showlegend=False
    )
    fig.show()

    # Dynamic interpretation
    print("\nTOP 5 MOST IMPORTANT FEATURES:")
    for i, row in importance_df.tail(5).iloc[::-1].iterrows():
        print(f"   {row['Feature']}: {row['Importance']:.4f}")


TOP 5 MOST IMPORTANT FEATURES:
   scheduled_shipping_days: 0.5845
   shipping_urgency: 0.1910
   shipping_mode_encoded: 0.1438
   type_encoded: 0.0227
   is_weekend: 0.0035


---

## 8. Save Best Model

**Talking Point**: "We save the trained model for deployment and future inference."

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================
import joblib
from pathlib import Path
from datetime import datetime

model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
model_path = model_dir / f'best_model_{timestamp}.pkl'

joblib.dump(best_model, model_path)
print(f"Model saved to: {model_path}")

# Save results summary
results_path = model_dir / f'training_results_{timestamp}.pkl'
results_to_save = {name: {k: v for k, v in res.items() if k != 'model' and k != 'y_proba'}
                   for name, res in results.items()}
joblib.dump(results_to_save, results_path)
print(f"Results saved to: {results_path}")

Model saved to: ../models/best_model_20251205_0058.pkl
Results saved to: ../models/training_results_20251205_0058.pkl


In [14]:
# Final Summary
print("\n" + "=" * 80)
print("MODEL TRAINING COMPLETE")
print("=" * 80)

print(f"""
TRAINING SUMMARY
{'='*60}

MODELS TRAINED: {len(results)}
{chr(10).join([f'   - {name}: F1={res["f1"]:.4f}' for name, res in results.items()])}

BEST MODEL: {best_model_name}
   Accuracy: {results[best_model_name]['accuracy']:.4f}
   F1 Score: {results[best_model_name]['f1']:.4f}
   Precision: {results[best_model_name]['precision']:.4f}
   Recall: {results[best_model_name]['recall']:.4f}
   ROC-AUC: {results[best_model_name]['roc_auc']:.4f}

BUSINESS METRICS:
   Late deliveries caught: {tp:,} out of {tp+fn:,} ({tp/(tp+fn)*100:.1f}%)
   False alarms: {fp:,} ({fp/(fp+tn)*100:.1f}% of on-time orders)

{'='*60}
Model saved to: {model_path}
Next: Run 05_business_impact.ipynb
""")


MODEL TRAINING COMPLETE

TRAINING SUMMARY

MODELS TRAINED: 6
   - Logistic Regression: F1=0.6871
   - Decision Tree: F1=0.7026
   - Random Forest: F1=0.6974
   - XGBoost: F1=0.7036
   - LightGBM: F1=0.6974
   - CatBoost: F1=0.6931

BEST MODEL: XGBoost
   Accuracy: 0.7074
   F1 Score: 0.7036
   Precision: 0.7441
   Recall: 0.7074
   ROC-AUC: 0.7823

BUSINESS METRICS:
   Late deliveries caught: 11,402 out of 19,796 (57.6%)
   False alarms: 2,170 (13.3% of on-time orders)

Model saved to: ../models/best_model_20251205_0058.pkl
Next: Run 05_business_impact.ipynb

